# LangGraph MCP - Fetch 튜토리얼

이 튜토리얼에서는 **Fetch** MCP 서버를 LangGraph와 통합하여 웹 페이지의 콘텐츠를 가져오고 LLM이 활용할 수 있는 마크다운으로 변환하는 방법을 학습합니다.

Fetch MCP 서버는 웹 페이지의 HTML을 **마크다운으로 변환**하여 LLM이 웹 콘텐츠를 효과적으로 이해하고 분석할 수 있게 합니다. URL을 입력하면 해당 페이지의 내용을 깔끔한 마크다운 형태로 반환합니다.

> 공식 저장소: [MCP Fetch Server](https://github.com/modelcontextprotocol/servers/tree/main/src/fetch)

## 학습 목표

1. Fetch MCP 서버의 역할과 동작 원리를 이해합니다
2. Fetch MCP 서버를 설치하고 설정하는 방법을 학습합니다
3. `fetch` 도구의 파라미터(`url`, `max_length`, `start_index`, `raw`)와 사용법을 이해합니다
4. `create_agent`를 사용하여 Fetch 기반 AI 에이전트를 생성합니다
5. ToolNode를 활용한 커스텀 워크플로우에 Fetch를 통합합니다
6. 실전 예제로 웹 콘텐츠 수집 및 분석을 수행합니다

## 목차

1. Fetch MCP 서버 소개
2. 환경 설정
3. Fetch MCP 서버 기본
4. fetch 도구 직접 활용
5. 페이지네이션 활용
6. Agent와 Fetch 통합
7. ToolNode 기반 커스텀 워크플로우
8. 실전 활용 예제

## 환경 설정

튜토리얼을 시작하기 전에 필요한 환경을 설정합니다. `dotenv`를 사용하여 API 키를 로드하고, `langchain_teddynote`의 로깅 기능을 활성화하여 LangSmith에서 실행 과정을 확인할 수 있도록 합니다.

LangSmith 추적을 활성화하면 에이전트의 추론 과정, 도구 호출, 응답 생성 등을 시각적으로 모니터링할 수 있어 디버깅에 큰 도움이 됩니다.

아래 코드는 환경 변수를 로드하고 LangSmith 프로젝트를 설정합니다.

In [ ]:
from dotenv import load_dotenv
from langchain_teddynote import logging

# 환경 변수 로드
load_dotenv(override=True)

# LangSmith 추적 프로젝트 이름 설정
logging.langsmith("LangGraph-V1-Tutorial")

---

## Part 1: Fetch MCP 서버 소개

### Fetch MCP 서버란?

Fetch MCP 서버는 웹 페이지의 콘텐츠를 가져와 **LLM이 처리하기 쉬운 마크다운 형식으로 변환**하는 MCP 서버입니다. 일반적으로 LLM은 HTML을 직접 처리하기 어렵고, 웹 페이지에 접근하는 능력이 없습니다. Fetch 서버는 이 문제를 해결합니다.

### Fetch가 해결하는 문제

| 기존 문제 | Fetch 해결책 |
|-----------|-------------|
| LLM이 웹 페이지에 직접 접근 불가 | URL을 통해 웹 콘텐츠를 가져와 제공 |
| HTML 태그가 섞인 비정형 데이터 | 깔끔한 마크다운으로 자동 변환 |
| 긴 웹 페이지 처리의 어려움 | `max_length`, `start_index`로 페이지네이션 지원 |
| robots.txt 제한으로 접근 불가 | `--ignore-robots-txt` 옵션으로 우회 가능 |

### 주요 특징

- **HTML → 마크다운 변환**: 웹 페이지의 HTML을 LLM 친화적인 마크다운으로 자동 변환
- **페이지네이션 지원**: `max_length`와 `start_index`로 긴 페이지를 분할하여 읽기
- **유연한 옵션**: User-Agent 설정, 프록시 지원, robots.txt 무시 등
- **raw 모드**: 마크다운 변환 없이 원본 콘텐츠 그대로 가져오기 가능
- **MCP 표준 준수**: MCP 프로토콜을 따르므로 어떤 MCP 클라이언트와도 호환

### Fetch MCP 도구

Fetch 서버는 단일 `fetch` 도구를 제공합니다:

#### `fetch`
URL에서 웹 콘텐츠를 가져와 마크다운으로 변환합니다.
- **`url`** (필수): 가져올 웹 페이지의 URL
- **`max_length`** (선택, 기본값: 5000): 반환할 콘텐츠의 최대 문자 수
- **`start_index`** (선택, 기본값: 0): 콘텐츠 시작 위치 (페이지네이션에 활용)
- **`raw`** (선택, 기본값: false): true이면 마크다운 변환 없이 원본 반환

### 설치 요구사항

Fetch 서버는 `uvx`를 통해 실행되므로 **uv**가 설치되어 있어야 합니다.

```bash
# uv 설치 확인
uv --version

# Fetch 서버 실행 테스트
uvx mcp-server-fetch --help
```

### 서버 실행 옵션

```bash
# 기본 실행
uvx mcp-server-fetch

# robots.txt 무시 (일부 사이트 접근용)
uvx mcp-server-fetch --ignore-robots-txt

# 커스텀 User-Agent 설정
uvx mcp-server-fetch --user-agent "MyBot/1.0"

# 프록시 서버 사용
uvx mcp-server-fetch --proxy-url http://proxy:8080
```

---

## Part 2: 기본 설정 및 패키지 임포트

Fetch MCP 서버를 사용하기 위해 필요한 패키지들을 임포트합니다. `langchain-mcp-adapters`는 MCP 서버를 LangChain 컴포넌트와 연결하는 핵심 라이브러리입니다.

```bash
# 필요한 패키지 설치 (이미 설치되어 있다면 생략)
uv add langchain-mcp-adapters
```

In [ ]:
import nest_asyncio
from typing import List, Dict, Any

from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

# MCP 클라이언트: 여러 MCP 서버에 연결하여 도구를 가져옵니다
from langchain_mcp_adapters.client import MultiServerMCPClient

# 비동기 호환을 활성화합니다 (Jupyter 환경에서 필요)
nest_asyncio.apply()

---

## Part 3: Fetch MCP 서버 기본

### MCP 클라이언트 설정 헬퍼 함수

`MultiServerMCPClient`를 사용하면 여러 MCP 서버에 동시에 연결하고 각 서버의 도구를 통합하여 사용할 수 있습니다. 아래는 Windows 환경과 Jupyter 호환성 패치와 함께 MCP 클라이언트를 설정하는 헬퍼 함수입니다.

**Windows 호환성 패치**: Windows 환경에서 Jupyter Notebook을 사용할 때 MCP stdio가 Jupyter의 `sys.stderr`를 서브프로세스에 전달하면서 `fileno()` 에러가 발생할 수 있습니다. 이를 방지하기 위한 패치를 적용합니다.

In [ ]:
import sys, os

# Windows + Jupyter workaround: MCP stdio passes Jupyter's sys.stderr to subprocess.Popen,
# but Jupyter's stderr doesn't support fileno(). Patch the default errlog to os.devnull.
if sys.platform == "win32":
    import mcp.client.stdio as _mcp_stdio

    _devnull_file = open(os.devnull, "w")

    # @asynccontextmanager wraps the original function — patch __wrapped__.__defaults__
    if hasattr(_mcp_stdio.stdio_client, "__wrapped__"):
        _mcp_stdio.stdio_client.__wrapped__.__defaults__ = (_devnull_file,)

    # Also patch the helper that creates the subprocess
    _mcp_stdio._create_platform_compatible_process.__defaults__ = (
        None,
        _devnull_file,
        None,
    )


async def setup_mcp_client(server_configs: dict):
    """MCP 클라이언트를 설정하고 도구를 가져옵니다.

    Args:
        server_configs: 서버 구성 딕셔너리. 각 서버의 이름을 키로,
                       연결 정보(command, args, transport 또는 url)를 값으로 가집니다.

    Returns:
        tuple: (MCP 클라이언트, 로드된 도구 목록)
    """
    # MCP 클라이언트 생성
    client = MultiServerMCPClient(server_configs)

    # 서버에 연결하여 도구 목록을 가져옵니다
    tools = await client.get_tools()

    # 로드된 도구 목록을 출력합니다
    print(f"[MCP] {len(tools)}개의 도구가 로드되었습니다:")
    for tool in tools:
        print(
            f"  - {tool.name}: {tool.description[:80]}..."
            if len(tool.description) > 80
            else f"  - {tool.name}: {tool.description}"
        )

    return client, tools

### Fetch MCP 서버 연결

Fetch MCP 서버는 `uvx`를 통해 실행됩니다. `stdio` 전송 방식을 사용하여 클라이언트가 자동으로 서버 프로세스를 관리합니다.

**서버 구성 설명:**
- `command`: `uvx` — Python 패키지 실행기
- `args`: `["mcp-server-fetch"]` — Fetch MCP 서버 실행
- `transport`: `stdio` — 표준 입출력을 통한 통신

**옵션 추가 시:**
- robots.txt 무시: `args`에 `"--ignore-robots-txt"` 추가
- 커스텀 User-Agent: `args`에 `"--user-agent"`, `"MyBot/1.0"` 추가

In [ ]:
import certifi

# Fetch MCP 서버 구성
fetch_server_config = {
    "fetch": {
        "command": "uvx",
        "args": ["mcp-server-fetch", "--ignore-robots-txt"],
        "transport": "stdio",
        "env": {"SSL_CERT_FILE": certifi.where()},
    }
}

# MCP 클라이언트 생성 및 도구 로드
client, tools = await setup_mcp_client(server_configs=fetch_server_config)

### 도구 상세 정보 확인

로드된 Fetch 도구의 이름, 설명, 파라미터 스키마를 상세히 확인합니다.

In [ ]:
import json

# 로드된 도구 상세 정보 출력
for i, tool in enumerate(tools):
    print(f"\n{'='*60}")
    print(f"도구 {i+1}: {tool.name}")
    print(f"{'='*60}")
    print(f"설명: {tool.description}")
    print(f"\n파라미터 스키마:")
    if hasattr(tool, "args_schema") and tool.args_schema:
        schema = (
            tool.args_schema.schema()
            if hasattr(tool.args_schema, "schema")
            else tool.args_schema
        )
        print(json.dumps(schema, indent=2, ensure_ascii=False))

---

## Part 4: fetch 도구 직접 활용

### fetch 도구란?

`fetch`는 URL에서 웹 콘텐츠를 가져와 마크다운으로 변환하는 도구입니다. 주요 파라미터는 다음과 같습니다:

| 파라미터 | 필수 | 기본값 | 설명 |
|----------|------|--------|------|
| `url` | ✅ | - | 가져올 웹 페이지 URL |
| `max_length` | ❌ | 5000 | 반환할 최대 문자 수 |
| `start_index` | ❌ | 0 | 콘텐츠 시작 위치 (페이지네이션) |
| `raw` | ❌ | false | true이면 원본 HTML 반환 |

아래 코드는 `fetch` 도구를 직접 호출하여 웹 페이지 콘텐츠를 가져옵니다.

In [ ]:
# fetch 도구 가져오기
fetch_tool = next(t for t in tools if t.name == "fetch")

# Python 공식 문서 페이지 가져오기 (기본 설정)
result = await fetch_tool.ainvoke({
    "url": "https://python.org",
})

print("Python.org 페이지 콘텐츠 (기본 설정):")
print("-" * 60)
# 결과가 길 수 있으므로 처음 2000자만 출력
print(result[:2000] if len(str(result)) > 2000 else result)

In [ ]:
# max_length를 사용하여 콘텐츠 길이 제한
result_short = await fetch_tool.ainvoke({
    "url": "https://docs.python.org/3/tutorial/index.html",
    "max_length": 1000,
})

print("Python 튜토리얼 페이지 (max_length=1000):")
print("-" * 60)
print(result_short)
print(f"\n반환된 콘텐츠 길이: {len(str(result_short))}자")

---

## Part 5: 페이지네이션 활용

### start_index를 활용한 페이지네이션

긴 웹 페이지를 한 번에 모두 가져오면 LLM의 컨텍스트 윈도우를 초과할 수 있습니다. `start_index`와 `max_length`를 조합하면 긴 페이지를 분할하여 순차적으로 읽을 수 있습니다.

**페이지네이션 원리:**
1. 첫 번째 호출: `start_index=0`, `max_length=3000` → 0~3000자
2. 두 번째 호출: `start_index=3000`, `max_length=3000` → 3000~6000자
3. 반복하여 전체 페이지 읽기

In [ ]:
# 페이지네이션 예제: 긴 페이지를 분할하여 읽기
target_url = "https://docs.python.org/3/tutorial/datastructures.html"

# 1단계: 처음 3000자 가져오기
chunk1 = await fetch_tool.ainvoke({
    "url": target_url,
    "max_length": 3000,
    "start_index": 0,
})

print("1단계 - 처음 3000자:")
print("-" * 60)
print(chunk1[:1500] if len(str(chunk1)) > 1500 else chunk1)
print(f"\n... (총 {len(str(chunk1))}자)")

In [ ]:
# 2단계: 3000자 이후부터 가져오기
chunk2 = await fetch_tool.ainvoke({
    "url": target_url,
    "max_length": 3000,
    "start_index": 3000,
})

print("2단계 - 3000자 이후:")
print("-" * 60)
print(chunk2[:1500] if len(str(chunk2)) > 1500 else chunk2)
print(f"\n... (총 {len(str(chunk2))}자)")

In [ ]:
# raw 모드 비교: 마크다운 변환 vs 원본 HTML

# 마크다운 변환 (기본)
md_result = await fetch_tool.ainvoke({
    "url": "https://example.com",
    "max_length": 2000,
})
print("📄 마크다운 변환 결과:")
print("-" * 60)
print(md_result)

print("\n" + "=" * 60 + "\n")

# 원본 HTML (raw=True)
raw_result = await fetch_tool.ainvoke({
    "url": "https://example.com",
    "max_length": 2000,
    "raw": True,
})
print("🔧 원본 HTML (raw=True):")
print("-" * 60)
print(raw_result)

---

## Part 6: Agent와 Fetch 통합

### create_agent 기반 Fetch 에이전트

`create_agent`는 LLM과 도구 목록을 전달하면 추론-행동(Reason-Act) 루프를 자동으로 구현하는 에이전트를 생성합니다. Fetch 도구를 에이전트에 통합하면, LLM이 필요에 따라 자동으로 웹 페이지를 가져와 내용을 분석합니다.

> 참고: LangGraph v1에서 기존의 `create_react_agent`는 deprecated 되었으며, `langchain.agents.create_agent`를 사용하는 것이 권장됩니다.

아래 코드는 Fetch 도구를 사용하는 에이전트를 생성하는 헬퍼 함수를 정의합니다.

In [ ]:
async def create_fetch_agent():
    """Fetch MCP 서버를 사용하는 에이전트를 생성합니다.

    Returns:
        CompiledStateGraph: 컴파일된 에이전트
    """
    # Fetch MCP 서버 구성
    server_configs = {
        "fetch": {
            "command": "uvx",
            "args": ["mcp-server-fetch"],
            "transport": "stdio",
            "env": {"SSL_CERT_FILE": certifi.where()},
        }
    }

    # MCP 클라이언트 생성 및 도구 로드
    client, tools = await setup_mcp_client(server_configs=server_configs)

    # LLM 설정
    llm = init_chat_model("claude-sonnet-4-6", temperature=0)

    # 에이전트 생성: Fetch 도구를 사용하는 에이전트
    agent = create_agent(
        llm,
        tools,
        checkpointer=InMemorySaver(),  # 대화 상태를 메모리에 저장
    )

    return agent

In [ ]:
# Fetch 에이전트 생성
agent = await create_fetch_agent()

In [ ]:
# 에이전트 그래프 구조 확인
agent

### 웹 페이지 내용 요약

에이전트가 Fetch 도구를 사용하여 웹 페이지를 가져오고 내용을 요약하는 과정을 확인합니다. 에이전트는 자동으로 `fetch` 도구를 호출하여 URL의 콘텐츠를 마크다운으로 변환한 후 분석합니다.

In [ ]:
from langchain_teddynote.messages import astream_graph, random_uuid
from langchain_core.runnables import RunnableConfig

# 대화 스레드 ID 설정
config = RunnableConfig(configurable={"thread_id": random_uuid()})

# 에이전트 실행: 웹 페이지 내용 요약 요청
response = await astream_graph(
    agent,
    inputs={
        "messages": [
            (
                "human",
                "https://python.org 페이지를 fetch로 가져와서 "
                "현재 Python의 최신 버전과 주요 뉴스를 요약해주세요.",
            )
        ]
    },
    config=config,
)

### 기술 블로그 내용 분석

Fetch 에이전트를 사용하여 기술 블로그 포스트의 내용을 가져와 핵심 내용을 분석합니다.

In [ ]:
# 새로운 대화 스레드 (독립적인 대화 시작)
config = RunnableConfig(configurable={"thread_id": random_uuid()})

# 기술 블로그 내용 분석
response = await astream_graph(
    agent,
    inputs={
        "messages": [
            (
                "human",
                "https://lilianweng.github.io/posts/2023-06-23-agent/ 페이지를 fetch로 가져와서 "
                "LLM 기반 에이전트의 핵심 구성 요소를 한국어로 정리해주세요.",
            )
        ]
    },
    config=config,
)

### 대화 컨텍스트 유지 (다중 턴 대화)

동일한 `thread_id`를 사용하면 이전 대화 내용이 유지됩니다. 같은 웹 페이지에 대해 연속으로 질문하면 에이전트가 이전 답변을 참조합니다.

In [ ]:
# 같은 thread_id로 연속 대화 (컨텍스트 유지)
config = RunnableConfig(configurable={"thread_id": random_uuid()})

# 첫 번째 질문: LangGraph 공식 문서 가져오기
await astream_graph(
    agent,
    inputs={
        "messages": [
            (
                "human",
                "https://langchain-ai.github.io/langgraph/ 페이지를 fetch로 가져와서 "
                "LangGraph의 주요 기능과 특징을 정리해주세요.",
            )
        ]
    },
    config=config,
)

In [ ]:
# 두 번째 질문: 이전 대화 컨텍스트를 유지하면서 후속 질문
await astream_graph(
    agent,
    inputs={
        "messages": [
            (
                "human",
                "위 내용을 바탕으로 LangGraph가 기존 LangChain과 어떻게 다른지, "
                "어떤 상황에서 LangGraph를 선택해야 하는지 설명해주세요.",
            )
        ]
    },
    config=config,  # 동일한 thread_id 사용으로 이전 대화 참조
)

---

## Part 7: ToolNode 기반 커스텀 워크플로우

### ToolNode와 Fetch

`ToolNode`를 사용하면 LangGraph에서 더 세밀한 제어가 가능한 커스텀 워크플로우를 만들 수 있습니다. `create_agent`와 달리, 그래프의 각 노드를 직접 정의하고 연결하여 복잡한 로직을 구현할 수 있습니다.

Fetch와 Tavily 검색 도구를 결합하면 **웹 페이지 콘텐츠 수집 + 웹 검색**을 동시에 활용하는 강력한 에이전트를 만들 수 있습니다.

### 워크플로우 구조

```
START → [agent 노드] → tools_condition → [tools 노드] → [agent 노드] → ...
                                        ↘ END (도구 호출 없을 때)
```

아래 코드는 Fetch + Tavily 검색을 결합한 커스텀 워크플로우를 정의합니다.

In [ ]:
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage
from typing import Annotated, TypedDict
from langchain_tavily import TavilySearch


class AgentState(TypedDict):
    """에이전트 상태 정의

    Attributes:
        messages: 대화 메시지 목록. add_messages 리듀서로 메시지가 누적됩니다.
    """

    messages: Annotated[List[BaseMessage], add_messages]


async def create_fetch_workflow():
    """Fetch + Tavily 검색을 결합한 커스텀 워크플로우를 생성합니다.

    Fetch 도구(웹 페이지 콘텐츠 수집)와 Tavily 도구(웹 검색)를 함께 사용하여
    특정 URL의 상세 내용과 관련 웹 검색 결과를 모두 활용할 수 있는 에이전트를 생성합니다.

    Returns:
        CompiledStateGraph: 컴파일된 워크플로우 그래프
    """
    # Fetch MCP 서버 구성
    server_configs = {
        "fetch": {
            "command": "uvx",
            "args": ["mcp-server-fetch"],
            "transport": "stdio",
            "env": {"SSL_CERT_FILE": certifi.where()},
        }
    }

    # MCP 클라이언트 생성 및 Fetch 도구 로드
    client, mcp_tools = await setup_mcp_client(server_configs=server_configs)

    # Tavily 웹 검색 도구 추가
    tavily_tool = TavilySearch(max_results=3)
    all_tools = mcp_tools + [tavily_tool]

    print(f"\n총 {len(all_tools)}개 도구 사용:")
    for t in all_tools:
        print(f"  - {t.name}")

    # LLM 설정 및 도구 바인딩
    llm = init_chat_model("claude-sonnet-4-6", temperature=0)
    llm_with_tools = llm.bind_tools(all_tools)

    # 워크플로우 그래프 생성
    workflow = StateGraph(AgentState)

    async def agent_node(state: AgentState):
        """에이전트 노드: LLM을 호출하여 응답을 생성합니다"""
        response = await llm_with_tools.ainvoke(state["messages"])
        return {"messages": [response]}

    # ToolNode 생성: 도구 호출을 처리합니다
    tool_node = ToolNode(all_tools)

    # 그래프에 노드 추가
    workflow.add_node("agent", agent_node)
    workflow.add_node("tools", tool_node)

    # 엣지 정의
    workflow.add_edge(START, "agent")
    # 조건부 엣지: 도구 호출 필요 시 tools로, 아니면 종료
    workflow.add_conditional_edges("agent", tools_condition)
    # 도구 실행 후 다시 에이전트로
    workflow.add_edge("tools", "agent")

    # 그래프 컴파일
    app = workflow.compile(checkpointer=InMemorySaver())

    return app

In [ ]:
# 커스텀 워크플로우 생성
workflow_app = await create_fetch_workflow()

In [ ]:
from IPython.display import Image, display

# 워크플로우 그래프 시각화
display(Image(workflow_app.get_graph().draw_mermaid_png()))

### Fetch + Tavily 복합 검색

Fetch로 특정 URL의 상세 내용을 가져오고, Tavily로 관련 최신 정보를 함께 검색합니다.

In [ ]:
# 새로운 대화 스레드 설정
config = RunnableConfig(configurable={"thread_id": random_uuid()})

# Fetch + Tavily 복합 검색
# 1. Fetch로 특정 페이지의 상세 내용 가져오기
# 2. Tavily로 관련 최신 정보 검색
await astream_graph(
    workflow_app,
    inputs={
        "messages": [
            (
                "human",
                "https://langchain-ai.github.io/langgraph/concepts/agentic_concepts/ 페이지를 fetch로 가져와서 내용을 분석하고, "
                "Tavily로 'LangGraph agentic workflow 2025' 관련 최신 정보도 검색해서 종합적으로 설명해주세요.",
            )
        ]
    },
    config=config,
)

---

## Part 8: 실전 활용 예제

### 예제 1: 여러 URL 비교 분석

여러 웹 페이지의 내용을 가져와서 비교 분석합니다. 에이전트가 각 URL을 순차적으로 fetch하여 내용을 종합합니다.

In [ ]:
# 새로운 에이전트 생성 (깨끗한 상태에서 시작)
agent = await create_fetch_agent()
config = RunnableConfig(configurable={"thread_id": random_uuid()})

# 여러 URL 비교: 서로 다른 프레임워크의 공식 페이지 분석
await astream_graph(
    agent,
    inputs={
        "messages": [
            (
                "human",
                "다음 두 페이지를 fetch로 각각 가져와서 비교해주세요:\n"
                "1. https://fastapi.tiangolo.com/ (FastAPI)\n"
                "2. https://flask.palletsprojects.com/ (Flask)\n\n"
                "각 프레임워크의 주요 특징, 장점, 사용 사례를 비교 분석해주세요.",
            )
        ]
    },
    config=config,
)

### 예제 2: 긴 문서 페이지네이션 읽기

에이전트에게 `start_index`를 활용한 페이지네이션으로 긴 문서를 분할하여 읽도록 요청합니다.

In [ ]:
config = RunnableConfig(configurable={"thread_id": random_uuid()})

# 긴 문서 페이지네이션으로 읽기
await astream_graph(
    agent,
    inputs={
        "messages": [
            (
                "human",
                "https://docs.python.org/3/tutorial/datastructures.html 페이지를 fetch로 가져오되, "
                "max_length=3000으로 설정해서 처음 부분을 먼저 읽고, "
                "그 다음 start_index=3000으로 이어서 읽어주세요. "
                "두 부분을 종합하여 Python 데이터 구조의 핵심 내용을 정리해주세요.",
            )
        ]
    },
    config=config,
)

### 예제 3: GitHub 저장소 정보 수집

Fetch를 사용하여 GitHub 저장소 페이지에서 프로젝트 정보를 수집하고 분석합니다.

In [ ]:
config = RunnableConfig(configurable={"thread_id": random_uuid()})

# GitHub 저장소 정보 수집 및 분석
await astream_graph(
    agent,
    inputs={
        "messages": [
            (
                "human",
                "https://github.com/langchain-ai/langgraph 페이지를 fetch로 가져와서 "
                "LangGraph 프로젝트의 소개, 주요 기능, 설치 방법, 시작 가이드를 정리해주세요.",
            )
        ]
    },
    config=config,
)

### 예제 4: API 문서 분석 및 코드 생성

Fetch로 API 문서를 가져와서 분석한 후, 실제 사용할 수 있는 코드 예제를 생성합니다.

In [ ]:
config = RunnableConfig(configurable={"thread_id": random_uuid()})

# API 문서 분석 및 코드 생성
await astream_graph(
    agent,
    inputs={
        "messages": [
            (
                "human",
                "https://docs.pydantic.dev/latest/concepts/models/ 페이지를 fetch로 가져와서 "
                "Pydantic v2의 Model 정의 방법을 분석하고, "
                "중첩 모델(nested model), 커스텀 검증(validator), JSON 직렬화를 포함한 "
                "실전 예제 코드를 작성해주세요.",
            )
        ]
    },
    config=config,
)

---

## 요약

이 튜토리얼에서는 Fetch MCP 서버를 LangGraph와 통합하는 전체 과정을 학습했습니다.

### 핵심 정리

| 개념 | 내용 |
|------|------|
| **Fetch MCP 서버** | 웹 페이지를 가져와 LLM 친화적인 마크다운으로 변환하는 MCP 서버 |
| **fetch 도구** | `url`, `max_length`, `start_index`, `raw` 파라미터로 웹 콘텐츠 수집 |
| **페이지네이션** | `start_index` + `max_length` 조합으로 긴 문서 분할 읽기 |
| **MCP 연결** | `uvx mcp-server-fetch` + stdio 전송 방식 |
| **에이전트 통합** | `create_agent`로 자동 도구 호출 루프 구현 |
| **커스텀 워크플로우** | `ToolNode` + `StateGraph`로 세밀한 흐름 제어 |

### Fetch 활용 팁

1. **적절한 max_length 설정**: 너무 크면 LLM 컨텍스트를 낭비하고, 너무 작으면 중요한 내용을 놓칠 수 있음. 기본값 5000이 대부분의 경우 적합
2. **페이지네이션 활용**: 긴 문서는 `start_index`를 활용하여 분할 읽기. 에이전트가 자동으로 판단하도록 요청 가능
3. **raw 모드 활용 시기**: HTML 구조 자체가 필요한 경우(스크래핑 패턴 분석 등)에만 `raw=True` 사용
4. **robots.txt 주의**: 일부 사이트는 robots.txt로 접근을 제한. 필요시 `--ignore-robots-txt` 옵션 사용
5. **Tavily와 조합**: Fetch(특정 URL 상세 내용) + Tavily(키워드 웹 검색) 조합으로 가장 포괄적인 정보 획득

### 다음 단계

- `06-MCP/` 폴더의 다른 MCP 튜토리얼을 통해 다양한 MCP 서버 활용법 학습
- [Smithery AI](https://smithery.ai/)에서 다양한 3rd Party MCP 서버 탐색
- Fetch + LangGraph + 사용자 정의 도구를 결합한 프로덕션 에이전트 구축